In [3]:
from pathlib import Path
import pandas as pd

In [4]:
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TRANSCRIPTS_PATH = REPO_ROOT / "data" / "processed" / "transcripts.csv"

assert TRANSCRIPTS_PATH.exists(), f"Missing: {TRANSCRIPTS_PATH}"

df = pd.read_csv(TRANSCRIPTS_PATH, parse_dates = ["date_parsed"])

# Day 1 retro safety asserts
assert df["ticker"].nunique() == 30, f"Expected 30 tickers, got {df['ticker'.nuinque()]}"
assert len(df) == 274, f"Expected 274 transcripts, got {len(df)}"

print(f"Shape: {df.shape}")
print(f"Columns + dtypes:\n{df.dtypes}")
print(f"Date range: {df['date_parsed'].min().date()} -> {df['date_parsed'].max().date()}")

Shape: (274, 5)
Columns + dtypes:
ticker                    str
quarter                   str
date_parsed    datetime64[us]
date_raw                  str
transcript                str
dtype: object
Date range: 2019-06-25 -> 2023-02-02


In [5]:
df["transcript_char_count"] = df["transcript"].str.len()
df["transcript_word_count"] = df["transcript"].str.split().str.len()

df[["transcript_char_count", "transcript_word_count"]].describe()

,transcript_char_count,transcript_word_count
count,274.000000,274.000000
mean,56803.204380,9649.521898
std,13365.426169,2247.247932
min,29067.000000,4910.000000
25%,50312.000000,8597.500000
50%,56270.000000,9530.000000
75%,61447.000000,10441.500000
max,188514.000000,32545.000000


In [7]:
df.groupby("ticker")["transcript_word_count"].agg(["count", "mean", "min", "max"]).sort_values("mean")

,count,mean,min,max
ticker,,,,
AMZN,10,6289.700000,5151,7519
ORCL,11,6532.818182,4910,7946
GOOGL,10,7910.200000,5747,9182
NFLX,9,8161.666667,7395,9178
PTON,10,8327.200000,6782,9597
NVDA,8,8525.125000,8159,9000
AAPL,14,8599.000000,8091,9279
PYPL,8,8713.750000,7482,9655
UNH,9,8920.000000,7916,10068


In [11]:
suspect = df.loc[df["transcript_word_count"].idxmax()]
print(f"Ticker: {suspect['ticker']}")
print(f"Quarter: {suspect['quarter']}")
print(f"Date: {suspect['date_parsed']}")
print(f"Word count: {suspect['transcript_word_count']:,}")
print(f"\nFirst 500 chars:\n{suspect['transcript'][:500]}")
print(f"\nLast 500 chars:\n{suspect['transcript'][-500:]}")

Ticker: WMT
Quarter: 2020-Q4
Date: 2021-02-18 00:00:00
Word count: 32,545

First 500 chars:
Prepared Remarks:
Dan Binder -- Vice President, Investor Relations
Good morning and welcome to Walmart's 2021 Investment Community Meeting. Thank you all for joining us on the webcast. We appreciate your interest in Walmart, I know the executive team looks forward to sharing their strategies with you and answering your questions. Now, let me get a few of our usual statements out of the way.
The information presented at today's meeting should be viewed in conjunction with our press release and ea

Last 500 chars:
- Guggenheim -- Analyst
Stephanie Wissink -- Jefferies & Co. -- Analyst
Michael Lasser -- UBS -- Analyst
Robert F. Ohmes -- Bank of America Merrill Lynch -- Analyst
Kelly Bania -- BMO -- Analyst
Oliver Chen -- Cowen and Company -- Analyst
Ed Yruma -- KeyBanc Capital Markets -- Analyst
Greg Melich -- Evercore ISI -- Analyst
Chuck Grom -- Gordon Haskett -- Analyst
Chris Horvers -- JPMorgan

In [ ]:
suspect_short = df.loc[df["transcript_word_count"].idxmin()]
print(f"Ticker: {suspect_short['ticker']}")
print(f"Quarter: {suspect_short['quarter']}")
print(f"Word count: {suspect_short['transcript_word_count']:,}")
print(f"\nFirst 500 chars:\n{suspect_short['transcript'][:500]}")

Ticker: ORCL
Quarter: 2021-Q4
Word count: 4,910

First 500 chars:
Prepared Remarks:
Operator
Welcome to Oracle's fourth-quarter 2021 earnings conference call. Now, I'd like to turn today's call over to Ken Bond, senior vice president. 
Ken Bond -- Senior Vice President
Thank you, Erica. Good afternoon, everyone, and welcome to Oracle's fourth-quarter and fiscal-year 2021 earnings conference call. A copy of the press release and financial tables, which includes a GAAP to non-GAAP reconciliation and other supplemental financial information, can be viewed and dow


In [14]:
# Other suspiciously long transcripts search

threshold = df["transcript_word_count"].quantile(0.95) # top 5%
print(f"95th percentile word count: {threshold:,.0f}")
print(f"Transcripts above 95th percentile: {(df['transcript_word_count'] > threshold).sum()}\n")

long_ones = df[df["transcript_word_count"] > threshold].sort_values("transcript_word_count", ascending = False)
print(long_ones[["ticker", "quarter", "date_parsed", "transcript_word_count"]].to_string())

95th percentile word count: 12,433
Transcripts above 95th percentile: 14

    ticker  quarter date_parsed  transcript_word_count
258    WMT  2020-Q4  2021-02-18                  32545
168    PFE  2021-Q4  2022-02-08                  17340
167    PFE  2021-Q3  2021-11-02                  14922
169    PFE  2022-Q2  2022-07-28                  14374
198   SBUX  2020-Q2  2020-04-28                  13893
164    PFE  2020-Q4  2021-02-02                  13767
166    PFE  2021-Q2  2021-07-28                  13603
171    PFE  2022-Q4  2023-01-31                  13403
265    XOM  2020-Q4  2021-02-02                  13305
39    BYND  2020-Q4  2021-02-25                  13099
30      BA  2019-Q2  2019-07-24                  12850
94     JNJ  2020-Q4  2021-01-26                  12727
206   SBUX  2022-Q4  2022-11-03                  12446
200   SBUX  2021-Q2  2021-04-27                  12440


In [15]:
for idx, row in long_ones.iterrows():
    print(f"\n{'='*60}")
    print(f"{row['ticker']} {row['quarter']} ({row['date_parsed'].date()}) - {row['transcript_word_count']:,} words")
    print(f"{'='*60}")
    print(row["transcript"][:300])


WMT 2020-Q4 (2021-02-18) - 32,545 words
Prepared Remarks:
Dan Binder -- Vice President, Investor Relations
Good morning and welcome to Walmart's 2021 Investment Community Meeting. Thank you all for joining us on the webcast. We appreciate your interest in Walmart, I know the executive team looks forward to sharing their strategies with yo

PFE 2021-Q4 (2022-02-08) - 17,340 words
Prepared Remarks:
Operator
Good day, everyone, and welcome to Pfizer's fourth quarter 2021 earnings conference call. Today's call is being recorded. At this time, I would like to turn the call over to Mr. Chris Stevo, senior vice president and chief investor relations officer.
Please go ahead, sir.


PFE 2021-Q3 (2021-11-02) - 14,922 words
Prepared Remarks:
Operator
Good day, everyone, and welcome to Pfizer's third quarter 2021 earnings conference call. Today's call is being recorded. At this time, I would like to turn the call over to Mr. Chris Stevo, senior vice president and chief investor relations officer

In [17]:
# drop walmart because earnings call mislabeled as investor day

before = len(df)
df = df[~((df["ticker"] == "WMT") & (df["quarter"] == "2020-Q4"))].copy()
after = len(df)
print(f"Dropped {before - after} row. Dataset: {before} -> {after} transcripts")

assert df["ticker"].nunique() == 30, "Lost a ticker"
assert len(df) == 273, f"Expected 273, got {len(df)}"
print("All 30 tickers still present.")


Dropped 0 row. Dataset: 273 -> 273 transcripts
All 30 tickers still present.


In [18]:
# Saving clean dataset as v2

processed_dir = REPO_ROOT / "data" / "processed"
output_path = processed_dir / "transcripts_v2.csv"

assert len(df) == 273, f"Refusing to save: expected 273 rows, got {len(df)}"
assert df["ticker"].nunique() == 30, "Refusing to save: not all 30 tickers present"

df.to_csv(output_path, index = False)

print(f"Saved: {output_path}")
print(f"Rows: {len(df)}, Tickers: {df['ticker'].nunique()}")
print(f"Date range: {df['date_parsed'].min().date()} -> {df['date_parsed'].max().date()}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")

Saved: d:\Projects\risk-radar\data\processed\transcripts_v2.csv
Rows: 273, Tickers: 30
Date range: 2019-06-25 -> 2023-02-02
File size: 14.7 MB
